# MACE + ASE: Model Inference and Geometry Relaxation

Demonstrates MACE interatomic potential model inference and structure relaxation using ASE.

## Steps

1. Install dependencies
2. Build a minimal MACE model with random weights
3. Run inference on a water molecule (H₂O) — compute energy & forces
4. Relax a distorted H₂O geometry with ASE BFGS and visualize convergence
5. Visualize structures before and after relaxation

## 1. Set Parameters

In [ ]:
MODEL_CONFIG = {
    "r_max": 5.0,
    "num_bessel": 8,
    "num_polynomial_cutoff": 5,
    "max_ell": 2,
    "num_interactions": 1,
    "num_elements": 4,
    "hidden_irreps": "16x0e + 16x1o",
    "MLP_irreps": "16x0e",
    "avg_num_neighbors": 5.0,
    "atomic_numbers": [1, 6, 7, 8],
    "atomic_energies": [-1.0, -3.0, -5.0, -7.0],
    "correlation": 2,
    "radial_type": "bessel",
}

H2O_POSITIONS_EQUILIBRIUM = [
    [0.0000,  0.0000,  0.1173],
    [0.0000,  0.7572, -0.4692],
    [0.0000, -0.7572, -0.4692],
]

H2O_POSITIONS_DISTORTED = [
    [0.0,  0.0,  0.0],
    [0.0,  0.9, -0.5],
    [0.0, -0.6, -0.6],
]

RELAXATION_PARAMETERS = {
    "FMAX": 0.1,
    "MAX_STEPS": 50,
}

## 2. Install Packages

In [ ]:
# !pip install torch "mace-torch" ase "e3nn==0.4.4" "numpy<=1.26.4" anywidget

## 3. Import Packages

In [ ]:
import numpy as np
import torch
# Adjust to satisfy e3nn requirements for safe serialization of slice objects
torch.serialization.add_safe_globals([slice])
from e3nn import o3
from ase import Atoms
from ase.calculators.calculator import Calculator, all_changes
from ase.optimize import BFGS
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from IPython.display import display
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase
from utils.visualize import visualize_materials, ViewersEnum
import mace.modules as mace_modules
from mace.modules import MACE
from mace.tools.torch_geometric.data import Data

## 4. Build MACE Model

In [ ]:
model = MACE(
    r_max=MODEL_CONFIG["r_max"],
    num_bessel=MODEL_CONFIG["num_bessel"],
    num_polynomial_cutoff=MODEL_CONFIG["num_polynomial_cutoff"],
    max_ell=MODEL_CONFIG["max_ell"],
    interaction_cls_first=mace_modules.interaction_classes["RealAgnosticResidualInteractionBlock"],
    interaction_cls=mace_modules.interaction_classes["RealAgnosticResidualInteractionBlock"],
    num_interactions=MODEL_CONFIG["num_interactions"],
    num_elements=MODEL_CONFIG["num_elements"],
    hidden_irreps=o3.Irreps(MODEL_CONFIG["hidden_irreps"]),
    MLP_irreps=o3.Irreps(MODEL_CONFIG["MLP_irreps"]),
    gate=torch.nn.functional.silu,
    atomic_energies=np.array(MODEL_CONFIG["atomic_energies"]),
    avg_num_neighbors=MODEL_CONFIG["avg_num_neighbors"],
    atomic_numbers=MODEL_CONFIG["atomic_numbers"],
    correlation=MODEL_CONFIG["correlation"],
    radial_type=MODEL_CONFIG["radial_type"],
)
model.double()
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"MACE model built: {n_params:,} parameters")

## 5. Run Inference on H₂O

Compute energy and forces for the equilibrium H₂O geometry.

In [ ]:
def build_h2o_graph(positions_list, atomic_numbers_map):
    """Build a fully-connected MACE Data graph for a small molecule."""
    n = len(positions_list)
    num_types = len(atomic_numbers_map)
    z_to_idx = {z: i for i, z in enumerate(atomic_numbers_map)}

    positions = torch.tensor(positions_list, dtype=torch.float64, requires_grad=True)

    node_attrs = torch.zeros(n, num_types, dtype=torch.float64)
    for i, z in enumerate([8, 1, 1]):
        node_attrs[i, z_to_idx[z]] = 1.0

    edges_src = [i for i in range(n) for j in range(n) if i != j]
    edges_dst = [j for i in range(n) for j in range(n) if i != j]
    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    n_edges = edge_index.shape[1]

    return Data(
        positions=positions,
        node_attrs=node_attrs,
        edge_index=edge_index,
        shifts=torch.zeros(n_edges, 3, dtype=torch.float64),
        unit_shifts=torch.zeros(n_edges, 3, dtype=torch.float64),
        cell=torch.zeros(3, 3, dtype=torch.float64),
        batch=torch.zeros(n, dtype=torch.long),
        ptr=torch.tensor([0, n], dtype=torch.long),
    )


data = build_h2o_graph(H2O_POSITIONS_EQUILIBRIUM, MODEL_CONFIG["atomic_numbers"])
output = model(data.to_dict(), training=False)

energy = output["energy"].item()
forces = output["forces"].detach().numpy()

print(f"H₂O energy:  {energy:.6f} eV")
print("H₂O forces (eV/Å):")
for elem, f in zip(["O", "H", "H"], forces):
    print(f"  {elem}: [{f[0]:+.6f}, {f[1]:+.6f}, {f[2]:+.6f}]")
print(f"Force sum:   [{forces.sum(0)[0]:.2e}, {forces.sum(0)[1]:.2e}, {forces.sum(0)[2]:.2e}]  (≈ 0)")

## 6. ASE Calculator and BFGS Relaxation

### 6.1. Define MACE ASE Calculator

In [ ]:
class MACECalculator(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(self, mace_model, atomic_numbers_map, **kwargs):
        super().__init__(**kwargs)
        self.mace_model = mace_model
        self.mace_model.eval()
        self.z_to_idx = {z: i for i, z in enumerate(atomic_numbers_map)}
        self.num_types = len(atomic_numbers_map)

    def calculate(self, atoms=None, properties=("energy", "forces"), system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        n = len(self.atoms)
        positions = torch.tensor(self.atoms.positions, dtype=torch.float64, requires_grad=True)

        node_attrs = torch.zeros(n, self.num_types, dtype=torch.float64)
        for i, z in enumerate(self.atoms.numbers):
            node_attrs[i, self.z_to_idx[z]] = 1.0

        edges_src = [i for i in range(n) for j in range(n) if i != j]
        edges_dst = [j for i in range(n) for j in range(n) if i != j]
        edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
        n_edges = edge_index.shape[1]

        data = Data(
            positions=positions,
            node_attrs=node_attrs,
            edge_index=edge_index,
            shifts=torch.zeros(n_edges, 3, dtype=torch.float64),
            unit_shifts=torch.zeros(n_edges, 3, dtype=torch.float64),
            cell=torch.zeros(3, 3, dtype=torch.float64),
            batch=torch.zeros(n, dtype=torch.long),
            ptr=torch.tensor([0, n], dtype=torch.long),
        )
        result = self.mace_model(data.to_dict(), training=False)
        self.results["energy"] = result["energy"].item()
        self.results["forces"] = result["forces"].detach().numpy()

### 6.2. Run Relaxation with Live Energy Plot

In [ ]:
calc = MACECalculator(model, MODEL_CONFIG["atomic_numbers"])

atoms_initial = Atoms("OH2", positions=H2O_POSITIONS_DISTORTED)
atoms_initial.calc = calc

atoms_relaxed = atoms_initial.copy()
atoms_relaxed.calc = MACECalculator(model, MODEL_CONFIG["atomic_numbers"])

steps, energies, max_forces = [], [], []

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Energy (eV)", "Max |F| (eV/Å)"),
)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="Energy"), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="Max |F|", line_color="orange"), row=1, col=2)
fig.update_layout(title_text="BFGS Relaxation — H₂O (distorted)", height=400)
fig_widget = go.FigureWidget(fig)
display(fig_widget)


def relaxation_callback():
    step = opt.nsteps
    e = atoms_relaxed.get_potential_energy()
    f = atoms_relaxed.get_forces()
    steps.append(step)
    energies.append(e)
    max_forces.append(np.max(np.abs(f)))
    print(f"  Step {step:3d}:  E = {e:.6f} eV,  max|F| = {max_forces[-1]:.6f} eV/Å")
    with fig_widget.batch_update():
        fig_widget.data[0].x = steps
        fig_widget.data[0].y = energies
        fig_widget.data[1].x = steps
        fig_widget.data[1].y = max_forces


e0 = atoms_relaxed.get_potential_energy()
f0 = atoms_relaxed.get_forces()
print(f"Initial:  E = {e0:.6f} eV,  max|F| = {np.max(np.abs(f0)):.6f} eV/Å")

opt = BFGS(atoms_relaxed, logfile=None)
opt.attach(relaxation_callback, interval=1)
opt.run(fmax=RELAXATION_PARAMETERS["FMAX"], steps=RELAXATION_PARAMETERS["MAX_STEPS"])

e1 = atoms_relaxed.get_potential_energy()
f1 = atoms_relaxed.get_forces()
print(f"\nFinal ({opt.nsteps} steps):  E = {e1:.6f} eV,  max|F| = {np.max(np.abs(f1)):.6f} eV/Å")
print(f"Energy change: {e1 - e0:.6f} eV")

## 7. Visualize Structures Before and After Relaxation

In [ ]:
VISUALIZATION_CELL = [[10, 0, 0], [0, 10, 0], [0, 0, 10]]


def atoms_to_material(atoms, title):
    atoms_copy = atoms.copy()
    atoms_copy.cell = VISUALIZATION_CELL
    atoms_copy.pbc = True
    material = Material.create(from_ase(atoms_copy))
    material.name = title
    return material


material_before = atoms_to_material(atoms_initial, f"Before relaxation  E={e0:.4f} eV")
material_after = atoms_to_material(atoms_relaxed, f"After {opt.nsteps} BFGS steps  E={e1:.4f} eV")

visualize_materials(
    [
        {"material": material_before, "title": material_before.name},
        {"material": material_after, "title": material_after.name},
    ],
    viewer=ViewersEnum.wave,
)

## 8. Summary

In [ ]:
print("=" * 50)
print(f"MACE model parameters: {n_params:,}")
print(f"Equilibrium H₂O energy: {energy:.6f} eV")
print(f"Relaxation:  {opt.nsteps} BFGS steps")
print(f"  Initial energy: {e0:.6f} eV")
print(f"  Final   energy: {e1:.6f} eV")
print(f"  Delta:          {e1 - e0:.6f} eV")
print(f"  Final max|F|:   {np.max(np.abs(f1)):.6f} eV/Å")
print("=" * 50)